In [1]:
import pyautogui
import pytesseract
import pygetwindow as gw
import time
from PIL import Image, ImageDraw, ImageFont
from models.captura_tela import CapturaTela

# Configuração do Tesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [2]:
# Cria uma instância da classe CapturaTela, que é responsável por capturar a tela e interagir com as janelas.
# Criei um script para capturar as telas, para reutilizar o código e evitar duplicação. O script está em models/captura_tela.py
janela_modulos = CapturaTela()

# Aguarda a janela Import abrir
if janela_modulos.aguardar_janela('Módulos', timeout=10):
    janela_modulos.focar_janela()
    # A linha abaixo está comentada para evitar criar arquivos de imagem desnecessários durante os testes. Descomente quando quiser salvar a captura da tela.
    #screenshot = janela_modulos.capturar(salvar=True, nome_arquivo='tela_modulos.png')

⏳ Aguardando janela com 'Módulos' aparecer...
🔍 Procurando janela com 'Módulos'...
✓ Janela encontrada: Módulos (Usuário: Claudinei Alsisi / Último Acesso: 11/02/2026 13:55:25) - \\Remote
  📍 Posição: (188, 18)
  📏 Tamanho: 990x676
✓ Janela 'Módulos' apareceu!
✓ Janela focada


In [3]:
# Busca janela com nome MÓDULOS
#janelas = gw.getWindowsWithTitle('Módulos')
#se encontrou algo, gw é um array de dados de janela
#if janelas:
#    janela = janelas[0]
#    print(f"✓ Janela encontrada: {janela.title}")
#    print(f"  Posição: ({janela.left}, {janela.top})")
#    print(f"  Tamanho: {janela.width}x{janela.height}")
#else:
#    print("❌ Janela não encontrada")
#    print("Janelas abertas:")
    # se não encontrou "MÓDULOS", printa todos os nomes das janelas abertas
#    for j in gw.getAllTitles():
#        if j.strip():
#            print(f"  - {j}")


In [4]:
# Foca e captura
# NECESSITA ATENÇÃO, SÓ FUNCIONA SE A JANELA ESTIVER VISÍVEL E FOCADA, POIS O PYAUTOGUI SÓ CAPTURA O QUE ESTÁ NA TELA
# SE A JANELA ESTIVER MINIMIZADA, O PYAUTOGUI VAI CAPTURAR A TELA DE FUNDO, NÃO A JANELA DO ONESOURCE
# ** PRECISO REVER ESTE COMANDO POSTERIORMENTE **
"""""
janela_modulos.activate()
time.sleep(1)  # Espera a janela focar

# Captura os dados de posição e tamanho da janela
screenshot = pyautogui.screenshot(region=(
    janela_modulos.left,
    janela_modulos.top,
    janela_modulos.width,
    janela_modulos.height
))
# apenas para visualização, pode ser removido posteriormente
print("✓ Screenshot capturado")
"""""

'""\njanela_modulos.activate()\ntime.sleep(1)  # Espera a janela focar\n\n# Captura os dados de posição e tamanho da janela\nscreenshot = pyautogui.screenshot(region=(\n    janela_modulos.left,\n    janela_modulos.top,\n    janela_modulos.width,\n    janela_modulos.height\n))\n# apenas para visualização, pode ser removido posteriormente\nprint("✓ Screenshot capturado")\n'

In [5]:
# 4. Processa OCR para encontrar "Import"
# O Tesseract retorna um dicionário com várias informações, incluindo o texto detectado e a confiança de cada palavra
print("\n🔍 Procurando texto 'Import' com OCR...")
# O parâmetro output_type=pytesseract.Output.DICT faz com que o resultado seja um dicionário, facilitando a análise
data = pytesseract.image_to_data(
    janela_modulos,
    lang='por',
    output_type=pytesseract.Output.DICT
)

# 5. Procura "Import" nos textos detectados
import_encontrado = False
# O Tesseract pode detectar várias palavras, então iteramos por todas elas para encontrar "Import"
texto_busca = "Import"
# O Tesseract retorna uma lista de palavras detectadas, suas posições e a confiança de cada detecção
for i in range(len(data['text'])):
    texto = data['text'][i].strip()
    # A confiança é um valor entre 0 e 100 que indica a certeza do Tesseract sobre a detecção da palavra
    conf = int(data['conf'][i])

    # Verifica se encontrou "Import" com boa confiança
    if texto_busca.lower() in texto.lower() and conf > 30:
        # Posições relativas (dentro da screenshot)
        x_rel = data['left'][i] + data['width'][i] // 2
        y_rel = data['top'][i] + data['height'][i] // 2

        # Converte para posições ABSOLUTAS na tela
        x_abs = janela.left + x_rel
        y_abs = janela.top + y_rel

        print(f"✓ Texto encontrado: '{texto}' (confiança: {conf}%)")
        print(f"  Posição relativa: ({x_rel}, {y_rel})")
        print(f"  Posição absoluta: ({x_abs}, {y_abs})")

        # 6. Clica no Import
        print(f"\n🖱️  Clicando em 'Import' em 2 segundos...")
        # O sleep é importante para garantir que o sistema tenha tempo de processar o clique, especialmente se a aplicação for lenta ou se houver animações
        time.sleep(2)
        # O comando click do pyautogui clica na posição absoluta calculada, que deve ser o centro da palavra "Import" detectada
        # Usei 2 comandos de click para garantir que o clique seja registrado, às vezes um clique pode ser perdido dependendo do sistema e da aplicação
        # sleep para dar um pequeno intervalo entre os cliques, aumentando a chance de o clique ser registrado corretamente
        pyautogui.click(x_abs, y_abs)
        time.sleep(0.1)
        pyautogui.click(x_abs, y_abs)
        # print de controle para indicar que o clique foi executado, pode ser removido posteriormente
        print("✓ Clique executado!")
        # Se encontrou "Import", não precisa continuar procurando, então quebramos o loop
        import_encontrado = True
        break
# Se não encontrou "Import", exibe os textos detectados para ajudar na depuração
if not import_encontrado:
    print("❌ Texto 'Import' não foi encontrado pelo OCR")
    print("\n📋 Textos detectados:")
    for texto in data['text']:
        if texto.strip():
            print(f"  - {texto}")
# Quebra de linha para separar a saída
print("\n" + "="*60)


🔍 Procurando texto 'Import' com OCR...


TypeError: Unsupported image object

In [ ]:
janela_Import = CapturaTela()

# Aguarda a janela Import abrir
if janela_Import.aguardar_janela('ONESOURCE GLOBAL TRADE - Import', timeout=10):
    janela_Import.focar_janela()
    #screenshot = janela_Import.capturar(salvar=True, nome_arquivo='tela_import.png')